# 05 — Domain Models
Instantiation examples for every model in `src/domain/models.py`.

In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
from src.domain.models import (
    UserContext,
    InvestigationRequest,
    PlanStep,
    ToolResult,
    TraceEvent,
    InvestigationTrace,
    AgentResult,
    StepStatus,
    EventType,
)

## UserContext

In [3]:
user = UserContext(
    user_id="analyst-001",
    session_id="sess-abc123",
    metadata={"team": "risk", "source": "notebook"},
)
print(user)

UserContext(user_id='analyst-001', session_id='sess-abc123', metadata={'team': 'risk', 'source': 'notebook'})


## InvestigationRequest

In [4]:
import uuid

request = InvestigationRequest(
    entity_name="TELEFONICA O2 HOLDINGS LIMITED",
    context=user,
    request_id=str(uuid.uuid4()),
    focus_areas=["ownership", "address_risk"],
    max_depth=4,
)
print(request.request_id)
print(request.entity_name)
print(request.focus_areas)
print(request.created_at)

24fdf8e9-0d61-4caf-8e1b-a982391400e6
TELEFONICA O2 HOLDINGS LIMITED
['ownership', 'address_risk']
2026-03-23 08:20:58.388484+00:00


## PlanStep

In [5]:
step1 = PlanStep(
    step_id="step_1",
    tool_name="get_direct_owners",
    description="Fetch immediate owners of the target company.",
    parameters={"company_name": request.entity_name},
)

step2 = PlanStep(
    step_id="step_2",
    tool_name="get_company_address_context",
    description="Retrieve the registered address.",
    parameters={"company_name": request.entity_name},
    depends_on=["step_1"],
)

print(step1)
print()
print(f"step_2 depends on: {step2.depends_on}")
print(f"step_2 status:     {step2.status}")

PlanStep(step_id='step_1', tool_name='get_direct_owners', description='Fetch immediate owners of the target company.', parameters={'company_name': 'TELEFONICA O2 HOLDINGS LIMITED'}, depends_on=[], status=<StepStatus.PENDING: 'pending'>, result=None)

step_2 depends on: ['step_1']
step_2 status:     StepStatus.PENDING


## ToolResult — success and failure

In [6]:
ok_result = ToolResult(
    tool_name="get_direct_owners",
    success=True,
    data=[{"owner_name": "Telefonica S A", "ownership_pct_min": 75, "ownership_pct_max": 100}],
    duration_ms=43.2,
)

err_result = ToolResult(
    tool_name="get_company_address_context",
    success=False,
    error="ServiceUnavailable: Neo4j unreachable",
    duration_ms=1200.0,
)

# Attach result to step and advance status
step1.result = ok_result
step1.status = StepStatus.SUCCESS

print(f"step_1 status : {step1.status}")
print(f"step_1 data   : {step1.result.data}")
print(f"err duration  : {err_result.duration_ms} ms")

step_1 status : StepStatus.SUCCESS
step_1 data   : [{'owner_name': 'Telefonica S A', 'ownership_pct_min': 75, 'ownership_pct_max': 100}]
err duration  : 1200.0 ms


## TraceEvent

In [7]:
e1 = TraceEvent(
    event_type=EventType.STEP_STARTED,
    step_id="step_1",
    message="Starting get_direct_owners for TELEFONICA O2 HOLDINGS LIMITED",
)

e2 = TraceEvent(
    event_type=EventType.STEP_COMPLETED,
    step_id="step_1",
    message="get_direct_owners returned 1 owner",
    payload={"owner_count": 1, "duration_ms": 43.2},
)

e3 = TraceEvent(
    event_type=EventType.STEP_FAILED,
    step_id="step_2",
    message="Address lookup failed",
    payload={"error": err_result.error},
)

for e in [e1, e2, e3]:
    print(f"[{e.timestamp:%H:%M:%S}] {e.event_type:30s}  step={e.step_id}  {e.message}")

[08:20:58] EventType.STEP_STARTED          step=step_1  Starting get_direct_owners for TELEFONICA O2 HOLDINGS LIMITED
[08:20:58] EventType.STEP_COMPLETED        step=step_1  get_direct_owners returned 1 owner
[08:20:58] EventType.STEP_FAILED           step=step_2  Address lookup failed


## InvestigationTrace

In [8]:
trace = InvestigationTrace(request_id=request.request_id)
for event in [e1, e2, e3]:
    trace.add(event)

print(f"Total events     : {len(trace.events)}")
print(f"Events for step_1: {len(trace.events_for_step('step_1'))}")
print(f"Failed steps     : {trace.failed_steps()}")

Total events     : 3
Events for step_1: 2
Failed steps     : ['step_2']


## AgentResult — success and failure

In [9]:
result = AgentResult(
    request_id=request.request_id,
    entity_name=request.entity_name,
    success=True,
    summary="Telefonica O2 Holdings Limited is wholly owned by Telefonica S.A. (75–100%). "
            "Address lookup failed due to connectivity.",
    findings={
        "ownership": ok_result.data,
        "address_risk": None,
    },
    trace=trace,
    tools_used=["get_direct_owners"],
)

print(f"success    : {result.success}")
print(f"summary    : {result.summary}")
print(f"findings   : {list(result.findings.keys())}")
print(f"trace      : {len(result.trace.events)} events")
print(f"tools_used : {result.tools_used}")

success    : True
summary    : Telefonica O2 Holdings Limited is wholly owned by Telefonica S.A. (75–100%). Address lookup failed due to connectivity.
findings   : ['ownership', 'address_risk']
trace      : 3 events
tools_used : ['get_direct_owners']


In [10]:
# Failure case
failed_result = AgentResult(
    request_id=request.request_id,
    entity_name=request.entity_name,
    success=False,
    error="Neo4j unreachable — investigation aborted.",
    trace=trace,
)
print(f"success    : {failed_result.success}")
print(f"error      : {failed_result.error}")
print(f"tools_used : {failed_result.tools_used}")  # [] — no tools called before failure

success    : False
error      : Neo4j unreachable — investigation aborted.
tools_used : []
